<a href="https://colab.research.google.com/github/nuna-aa/paper-reproduction/blob/main/tokenization/fairness/multillingual-tokenization-parity.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install langcodes language_data

In [ ]:
# Import required dependencies

from transformers import AutoTokenizer
from datasets import load_dataset, get_dataset_config_names
from concurrent.futures import ThreadPoolExecutor, as_completed
from functools import reduce
from tqdm import tqdm
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import huggingface_hub
import langcodes
import os

In [ ]:
huggingface_hub.login()

> **Note:** Although FLORES-200 covers 219 languages, many evaluated models were not trained on all of them. Languages absent from tokenizer training data lack dedicated subword vocabulary and therefore undergo character-level or byte-level segmentation, resulting in substantially higher tokenization premiums. NLLB-200 differs because it was trained with explicit objectives covering all FLORES languages, allowing its tokenizer to learn balanced subword representations and produce consistently low premiums across the benchmark.

| Model | Tokenizer | Vocab | Training Data | Languages | Purpose |
|-------|-----------|-------|---------------|-----------|---------|
| NLLB-200 | SentencePiece BPE | 256,204 | Parallel translation data, 200 languages | 200 | Machine translation research, especially low-resource languages |
| Gemma3 | SentencePiece with byte-level encodings | 262k | Multilingual web data | 140+ | General purpose LM |
| Qwen3 | Byte-level BPE | 151,669 | 36 trillion tokens across 119 languages | 119 | General purpose LM |
| Tiny-Aya | SentencePiece BPE | 262k | 6T tokens, fairness-focused multilingual design | 70+ | Multilingual LM |

In [ ]:
# ─────────────────────────────────────────────
# CONFIG
# ─────────────────────────────────────────────

RAW_CACHE     = "flores_plus_raw.parquet"
RESULTS_FILE  = "tokenization_results.csv"
UNK_THRESHOLD = 0.10          # exclude language if UNK rate > 10%
FIGURES_DIR   = "figures"
ENGLISH_LANG_CODE = "eng_Latn_stan1293"
os.makedirs(FIGURES_DIR, exist_ok=True)

TOKENIZER_IDS = {
    "tiny-aya": "CohereLabs/tiny-aya-global",
    "qwen3":    "Qwen/Qwen3-0.6B",
    "gemma3":   "google/gemma-3-1b-it",
    "meta":     "facebook/nllb-200-distilled-600M",
}


In [ ]:
# ─────────────────────────────────────────────
# 1. DATA
# ─────────────────────────────────────────────
def load_flores() -> pd.DataFrame:
    """
    Returns a long DataFrame with columns: id, lang_code, text
    One row per sentence per language — no pivot.
    Cached to parquet after first download.
    """
    if os.path.exists(RAW_CACHE):
      print(f"[cache] Loading raw data from {RAW_CACHE}")
      return pd.read_parquet(RAW_CACHE)

    print(f"[data] Downloading FLORES+ devtest (all languages)")

    ds =  load_dataset("openlanguagedata/flores_plus", split="devtest")
    df = ds.to_pandas()

    df["lang_code"] = (
    df["iso_639_3"] + "_" +
    df["iso_15924"] + "_" +
    df["glottocode"]
   )

    # Append variant only where it exists and is non-empty
    mask = df["variant"].notna() & (df["variant"].str.strip() != "")
    df.loc[mask, "lang_code"] = df.loc[mask, "lang_code"] + "_" + df.loc[mask, "variant"].str.strip()

    df = df[["id", "lang_code", "text"]].copy()

    print(f"[data] Rows: {len(df):,}  |  Languages: {df['lang_code'].nunique()}")

    df.to_parquet(RAW_CACHE)
    print(f"[data] Cached -> {RAW_CACHE}")

    return df

In [ ]:
def get_aligned_pair(df: pd.DataFrame, lang_code: str):
    """
    Returns (eng_sentences, lang_sentences) as lists, aligned by sentence id.
    Join happens on demand — no wide pivot table in memory.

    Both english and lang are Series indexed by sentence id, but they might not have the same ids.
    Some sentences could be missing in one language but present in the other.
    The concat with dropna() keeps only the ids that exist in both.
    """

    english = df[df["lang_code"] == ENGLISH_LANG_CODE].set_index("id")["text"]

    if lang_code == ENGLISH_LANG_CODE:
        # If the target language is English, align English with English
        return english.tolist(), english.tolist()
    else:
        lang    = df[df["lang_code"] == lang_code ].set_index("id")["text"]
        paired  = pd.concat([english, lang], axis=1, keys=[ENGLISH_LANG_CODE, lang_code]).dropna()
        return paired[ENGLISH_LANG_CODE].tolist(), paired[lang_code].tolist()

In [ ]:
# ─────────────────────────────────────────────
# 2. TOKENIZER HELPERS
# ─────────────────────────────────────────────

def load_tokenizers() -> dict:
  tokenizers = {}
  for name, model_id in TOKENIZER_IDS.items():
    print(f"[tokenizer] Loading {name} from {model_id}")
    tokenizer = AutoTokenizer.from_pretrained(model_id)
    tokenizers[name] = tokenizer
    vocab_size =getattr(tokenizer, "vocab_size", "?")
    print(f"vocab_size = {vocab_size:,}" if isinstance(vocab_size, int) else f"vocab_size = {vocab_size}")

  return tokenizers


In [ ]:
def tokenize(tokenizer, sentence: str) -> int:
  return tokenizer.encode(sentence, add_special_tokens=False)

In [ ]:
def token_count(tokenizer, sentence: str) -> int:
  return len(tokenize(tokenizer, sentence))

In [ ]:
def unk_rate(tokenizer, token_ids: list[int]) -> float:
  unk_id = tokenizer.unk_token_id
  if unk_id is None:
    return 0.0

  total = unks = 0

  total += len(token_ids)
  unks += token_ids.count(unk_id)

  return unks / total if total > 0 else 0.0

In [ ]:
# ─────────────────────────────────────────────
# 3. PREMIUM COMPUTATION
# ─────────────────────────────────────────────

def compute(df: pd.DataFrame, tokenizers: dict) -> pd.DataFrame:
  """
    For every (model, language) pair:
      - Check UNK rate; mark excluded if > UNK_THRESHOLD
      - Compute per-sentence premium = n_tokens_lang / n_tokens_eng
      - Aggregate: mean, median, std premium per language
    Returns one row per (model, language).
   """

  all_languages = df["lang_code"].unique()
  print(f"\n[compute] {len(all_languages)} languages x {len(tokenizers)} models\n")

  records = []

  for model_name, tokenizer in tokenizers.items():
    print(f"--{model_name}")

    for i, lang in enumerate(all_languages):

      eng_sentences, lang_sentences = get_aligned_pair(df=df, lang_code=lang)

      if not eng_sentences or not lang_sentences:
        continue


      script = lang.split("_")[1] if "_" in lang else "Unknown"

      # UNK Filter
      # If the language is English, its UNK rate would also be calculated here.
      unk = unk_rate(tokenizer, [token for sent in lang_sentences for token in tokenize(tokenizer,sent)])
      if unk > UNK_THRESHOLD:
          records.append({
                    "model":          model_name,
                    "language":       lang,
                    "script":         script,
                    "excluded":       True,
                    "unk_rate":       round(unk, 4),
                    "mean_premium":   None,
                    "median_premium": None,
                    "std_premium":    None,
                    "n_sentences":    len(lang_sentences),
          })
          continue

      # Per-sentence Premium

      premiums = []

      for eng_sent, lang_sent in zip(eng_sentences, lang_sentences):
          eng_token_counts = token_count(tokenizer, eng_sent)
          lang_token_counts = token_count(tokenizer, lang_sent)

          if eng_token_counts > 0:
              premiums.append(lang_token_counts/ eng_token_counts)


      if not premiums:
        continue

      # Aggregate
      records.append({
                "model":          model_name,
                "language":       lang,
                "script":         script,
                "excluded":       False,
                "unk_rate":       round(unk, 4),
                "mean_premium":   round(float(np.mean(premiums)),   4),
                "median_premium": round(float(np.median(premiums)), 4),
                "std_premium":    round(float(np.std(premiums)),    4),
                "n_sentences":    len(premiums),
      })

      if (i + 1) % 25 == 0:
        print(f"{i + 1}/{len(all_languages)} languages processed")

    print(f"{len(all_languages)}/{len(all_languages)} languages done")

  results = pd.DataFrame(records)
  results.to_csv(RESULTS_FILE, index=False)
  print(f"\n[compute] Saved -> {RESULTS_FILE}")

  return results

In [ ]:
# ─────────────────────────────────────────────
# 4. VISUALIZATIONS
# ─────────────────────────────────────────────

PALETTE = {
    "tiny-aya": "#2196F3",
    #"qwen3":    "#4CAF50",
    #"smollm3":  "HuggingFaceTB/SmolLM3-3B",
    #"gemma3":   "google/gemma-3-1b-it",
}


def fig_heatmap(results: pd.DataFrame, lang_names: dict):
    """Heatmap: languages (rows) x models (cols), cell = mean premium."""
    active = results[~results["excluded"] & results["mean_premium"].notna()]
    pivot  = active.pivot_table(index="language", columns="model", values="mean_premium")
    pivot  = pivot.sort_values("tiny-aya", ascending=False)
    pivot.index = [lang_names.get(code, code) for code in pivot.index]  # ← map to names

    fig, ax = plt.subplots(figsize=(10, max(8, len(pivot) * 0.18)))
    sns.heatmap(
        pivot, ax=ax,
        cmap="YlOrRd", center=1.0, vmin=0.8, vmax=6.0,
        linewidths=0.3, linecolor="#ddd",
        cbar_kws={"label": "Mean Premium (relative to English)"},
    )
    ax.set_title("Tokenization Premium by Language & Model\n(1.0 = parity with English)", pad=14)
    ax.set_xlabel("Model")
    ax.set_ylabel("Language")
    ax.tick_params(axis="y", labelsize=6)
    plt.tight_layout()
    path = f"{FIGURES_DIR}/heatmap.png"
    fig.savefig(path, dpi=150)
    plt.close(fig)
    print(f"[fig] {path}")


def fig_boxplot_by_script(results: pd.DataFrame):
    """Box plots: premium distribution per script, one panel per model."""
    active = results[~results["excluded"] & results["mean_premium"].notna()].copy()

    # Only scripts with >= 3 languages
    script_counts = active.groupby("script")["language"].nunique()
    valid_scripts = script_counts[script_counts >= 3].index
    active = active[active["script"].isin(valid_scripts)]

    models = list(active["model"].unique())

    # Handle the case where there is only one model
    if len(models) == 1:
        fig, ax = plt.subplots(1, 1, figsize=(5 * len(models), 6), sharey=True)
        axes = [ax] # Wrap the single ax in a list to make it iterable
    else:
        fig, axes = plt.subplots(1, len(models), figsize=(5 * len(models), 6), sharey=True)

    for ax, model in zip(axes, models):
        subset = active[active["model"] == model]
        order  = (subset.groupby("script")["mean_premium"]
                        .median()
                        .sort_values(ascending=False)
                        .index)
        sns.boxplot(
            data=subset, x="mean_premium", y="script", order=order,
            ax=ax, color=PALETTE.get(model, "#999"), width=0.6,
        )
        ax.axvline(1.0, color="black", linewidth=0.8, linestyle="--", alpha=0.6)
        ax.set_title(model, fontsize=11, fontweight="bold")
        ax.set_xlabel("Mean Premium")
        ax.set_ylabel("Script" if ax == axes[0] else "")

    fig.suptitle("Premium Distribution by Script Group", fontsize=13, y=1.02)
    plt.tight_layout()
    path = f"{FIGURES_DIR}/boxplot_by_script.png"
    fig.savefig(path, dpi=150, bbox_inches="tight")
    plt.close(fig)
    print(f"[fig] {path}")

def fig_bar_all_languages(results: pd.DataFrame, lang_names: dict):
    """Horizontal bar chart: all languages sorted by premium, per model."""
    active = results[~results["excluded"] & results["mean_premium"].notna()]
    models = list(active["model"].unique())

    n = active["language"].nunique()  # total number of languages

    if len(models) == 1:
        fig, ax = plt.subplots(1, 1, figsize=(7, n * 0.2), sharey=False)
        axes = [ax]
    else:
        fig, axes = plt.subplots(1, len(models), figsize=(7 * len(models), n * 0.2), sharey=False)

    for ax, model in zip(axes, models):
        subset   = active[active["model"] == model].set_index("language")["mean_premium"].sort_values()
        colors   = ["#E91E63" if v > 2 else "#4CAF50" for v in subset.values]
        labels   = [lang_names.get(code, code) for code in subset.index]

        ax.barh(labels, subset.values, color=colors, edgecolor="white", height=0.7)
        ax.axvline(1.0, color="black", linewidth=0.9, linestyle="--")
        ax.set_title(model, fontsize=11, fontweight="bold")
        ax.set_xlabel("Mean Premium")
        ax.tick_params(axis="y", labelsize=7)

    fig.suptitle("All Languages by Tokenization Premium", fontsize=13, y=1.01)
    plt.tight_layout()
    path = f"{FIGURES_DIR}/bar_all_languages.png"
    fig.savefig(path, dpi=150, bbox_inches="tight")
    plt.close(fig)
    print(f"[fig] {path}")


def fig_scatter_script(results: pd.DataFrame):
    """Scatter: one point per language per model, y=premium, colored by script."""
    active  = results[~results["excluded"] & results["mean_premium"].notna()].copy()
    scripts = sorted(active["script"].unique())
    cmap    = plt.cm.get_cmap("tab20", len(scripts))
    color_map = {s: cmap(i) for i, s in enumerate(scripts)}

    models  = list(active["model"].unique())
    model_x = {m: i for i, m in enumerate(models)}
    active["x"] = active["model"].map(model_x)

    fig, ax = plt.subplots(figsize=(10, 6))
    for script, grp in active.groupby("script"):
        jitter = np.random.uniform(-0.15, 0.15, len(grp))
        ax.scatter(
            grp["x"] + jitter, grp["mean_premium"],
            color=color_map[script], label=script,
            alpha=0.7, s=25, edgecolors="none",
        )

    ax.set_xticks(range(len(models)))
    ax.set_xticklabels(models)
    ax.axhline(1.0, color="black", linewidth=0.8, linestyle="--")
    ax.set_ylabel("Mean Premium (relative to English)")
    ax.set_title("Tokenization Premium per Language, Colored by Script")
    ax.legend(bbox_to_anchor=(1.01, 1), loc="upper left", fontsize=7, ncol=2)
    plt.tight_layout()
    path = f"{FIGURES_DIR}/scatter_by_script.png"
    fig.savefig(path, dpi=150, bbox_inches="tight")
    plt.close(fig)
    print(f"[fig] {path}")


def print_summary(results: pd.DataFrame, lang_names: dict):
    """Print summary statistics and top-10 worst languages per model."""
    active = results[~results["excluded"] & results["mean_premium"].notna()]

    summary = (active.groupby("model")["mean_premium"]
                     .agg(["mean", "median", "std", "min", "max", "count"])
                     .round(3))
    summary.columns = ["Mean", "Median", "Std", "Min", "Max", "N Languages"]
    print("\n-- Summary Statistics ------------------------------------------")
    print(summary.to_string())
    summary.to_csv(f"{FIGURES_DIR}/summary_stats.csv")

    print("\n-- Top 10 Highest Premium Languages ----------------------------")
    for model in active["model"].unique():
        top10 = (active[active["model"] == model]
                 .nlargest(10, "mean_premium")[["language", "script", "mean_premium"]]
                 .copy())
        top10["language"] = top10["language"].map(lambda c: lang_names.get(c, c))  # ← map to names
        print(f"\n  {model}:")
        print(top10.to_string(index=False))

    excluded = results[results["excluded"]].copy()
    if not excluded.empty:
        excluded["language"] = excluded["language"].map(lambda c: lang_names.get(c, c))  # ← map to names
        print(f"\n-- Excluded languages (UNK rate > {UNK_THRESHOLD:.0%}) ------------------")
        print(excluded[["model", "language", "unk_rate"]].to_string(index=False))


In [ ]:
# ─────────────────────────────────────────────
# 5. MAIN
# ─────────────────────────────────────────────

print("=" * 60)
print("  Tokenization Fairness  (arxiv:2305.15425)")
print("=" * 60)

df         = load_flores()
print()

  Tokenization Fairness  (arxiv:2305.15425)
[cache] Loading raw data from flores_plus_raw.parquet



In [ ]:
tokenizers = load_tokenizers()
print()

[tokenizer] Loading tiny-aya from CohereLabs/tiny-aya-global
vocab_size = 261,000
[tokenizer] Loading qwen3 from Qwen/Qwen3-0.6B
vocab_size = 151,643
[tokenizer] Loading gemma3 from google/gemma-3-1b-it
vocab_size = 262,144
[tokenizer] Loading meta from facebook/nllb-200-distilled-600M
vocab_size = 256,204



In [ ]:
results    = compute(df, tokenizers)


[compute] 219 languages x 4 models

--tiny-aya
25/219 languages processed
50/219 languages processed
75/219 languages processed
100/219 languages processed
125/219 languages processed
150/219 languages processed
175/219 languages processed
200/219 languages processed
219/219 languages done
--qwen3
25/219 languages processed
50/219 languages processed
75/219 languages processed
100/219 languages processed
125/219 languages processed
150/219 languages processed
175/219 languages processed
200/219 languages processed
219/219 languages done
--gemma3
25/219 languages processed
50/219 languages processed
75/219 languages processed
100/219 languages processed
125/219 languages processed
150/219 languages processed
175/219 languages processed
200/219 languages processed
219/219 languages done
--meta
25/219 languages processed
50/219 languages processed
75/219 languages processed
100/219 languages processed
125/219 languages processed
150/219 languages processed
175/219 languages processed
200

In [ ]:
def get_lang_name(lang_code: str) -> str:
    iso, script = lang_code.split("_")[0], lang_code.split("_")[1]
    try:
        name = langcodes.Language.get(iso).display_name()
        return f"{name} ({script})"
    except:
        return lang_code

lang_names = {code: get_lang_name(code) for code in results["language"].unique()}

In [ ]:
print("\n[figures] Generating ...")
fig_heatmap(results, lang_names=lang_names)
fig_boxplot_by_script(results)
fig_bar_all_languages(results, lang_names=lang_names)
fig_scatter_script(results)
print_summary(results, lang_names=lang_names)

print(f"\nDone.  Results -> '{RESULTS_FILE}'   Figures -> '{FIGURES_DIR}/'")


[figures] Generating ...
[fig] figures/heatmap.png
[fig] figures/boxplot_by_script.png
[fig] figures/bar_all_languages.png


/tmp/ipykernel_23979/1885199825.py:114: MatplotlibDeprecationWarning: The get_cmap function was deprecated in Matplotlib 3.7 and will be removed in 3.11. Use ``matplotlib.colormaps[name]`` or ``matplotlib.colormaps.get_cmap()`` or ``pyplot.get_cmap()`` instead.
  cmap    = plt.cm.get_cmap("tab20", len(scripts))


[fig] figures/scatter_by_script.png

-- Summary Statistics ------------------------------------------
           Mean  Median    Std    Min     Max  N Languages
model                                                     
gemma3    2.027   1.868  1.101  1.000  11.576          219
meta      1.377   1.340  0.241  0.934   2.547          216
qwen3     2.794   2.150  1.730  1.000  10.716          219
tiny-aya  2.691   1.854  2.554  1.000  16.922          219

-- Top 10 Highest Premium Languages ----------------------------

  tiny-aya:
                          language script  mean_premium
                   Dzongkha (Tibt)   Tibt       16.9219
                    Tibetan (Tibt)   Tibt       15.5453
             Halh Mongolian (Mong)   Mong       15.1019
                   Georgian (Geor)   Geor       13.3854
                    Santali (Olck)   Olck       12.6340
                    Sinhala (Sinh)   Sinh       11.7044
                       N’Ko (Nkoo)   Nkoo       11.1333
                 

# 6. Analysis


**1. Training Coverage Strongly Influences Tokenization Premium**

Models like:

`Gemma 3`

`Qwen3`

`Tiny Aya `

were trained on subset multilingual corpora, not all FLORES languages.

In contrast, NLLB-200 was explicitly trained on all 200 FLORES languages as translation targets.

*Consequence*

When a language is absent from tokenizer training data: its frequent morphemes are not included in the vocabulary tokenization falls back to character-level or byte-level segmentation


**2. Script Structure Also Explains the Clustering**


From the chart:

| Script                    | Typical Premium Pattern |
| ------------------------- | ----------------------- |
| Latin                     | Lowest                  |
| Devanagari                | Moderate                |
| Cyrillic                  | Moderate                |
| Arabic                    | Higher                  |
| Tibetan / Southeast Asian | Very High               |


This occurs because tokenizers trained primarily on Latin scripts learn merges for Latin morphemes, not for other orthographies.

**3. Vocabulary Size Is Not the Dominant Factor**

The results illustrate an important insight:

Large vocabularies do not guarantee fairness.

Example:

Gemma 3 has a very large vocabulary (~256k).

Yet the chart shows premiums up to ~11×.

Why?

Because vocabulary allocation is frequency-driven

Thus the vocabulary may not be evenly distributed across languages.

**4. Evidence of Tokenizer-Induced Inequality**

The scatter plot demonstrates the phenomenon discussed in the paper:

Language Model Tokenizers Introduce Unfairness Between Languages

Key empirical signal:

Latin languages cluster around 1–2×

minority scripts reach 10–17×

This confirms tokenization cost disparity.